In [ ]:
%pip install smolagents -U
%pip install ddgs

In [11]:
from huggingface_hub import login
import os

HF_Token = os.getenv("HF_TOKEN")
login(token=HF_Token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, Tool, tool, VisitWebpageTool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."
    
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.
    
    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    
    services: dict[str, float] = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    best_service: str = max(
        services,
        key=lambda service: services[service],
    )
    return best_service
    

In [8]:
class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""
    
    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }
    
    output_type = "string"
    
    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }
        
        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

In [10]:
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        FinalAnswerTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool()
    ],
    model = InferenceClientModel(),
    max_steps = 5,
    verbosity_level = 2
)

# Use a model supported by Hugging Face Inference Providers.
agent.model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=HF_Token,
)

result = agent.run(
    "Give me the best playlist for a party at Wayne's mansion. "
    "The party idea is a 'villain masquerade' theme."
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me the best playlist for a party at Wayne's mansion. The party idea is a 'villain masquerade' theme.       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use the `superhero_party_theme_generator` tool to generate a creative villain masquerade party     
theme, and then use the generated theme to create a playlist for the party at Wayne's mansion.                     
                                                                                                                   
<code>                                                                                                             
theme = superhero_party_theme_generator(category="villain masquerade")                                             
print(f"The generated theme is: {theme}")                                                                          
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme = superhero_party_theme_generator(category="villain masquerade")                                           
  print(f"The generated theme is: {theme}")                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The generated theme is: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 2.89 seconds| Input tokens: 2,308 | Output tokens: 74]

... (8 outputs left out)

Execution logs:
The best playlist URL for the party is: https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay

Final answer: https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay

[Step 3: Duration 4.24 seconds| Input tokens: 8,407 | Output tokens: 337]

https://open.spotify.com/playlist/3cejFigsE9RrSdG4xUCmay
